# Phase 8: resolution test, C6 at 512 pixels

**CRISP-DM stage:** Modelling and evaluation

This is the Phase 7 notebook `07_core_C6` with **one change: the image size is 512 instead of 256**, read from `resolution_test_size` in `configs/C6_patchcore_efficientnet.yaml`. Every other setting, the seeds, half A and the measuring code stay the same, so any difference comes from the size alone. C6 (PatchCore with EfficientNet-B0) was chosen as the best pair on half A, see DECISIONS.md.

Two secondary metrics for the appendix are also saved: image AUPRC and pixel average precision. Files are named `C6_512_...`, so they never mix with the 256 results.

**One product per notebook.** The timing test showed fitting takes about 10 times longer at 512, so Vial alone needs about 1.6 hours. The two products run as two notebooks side by side, `08_resolution_C6_sheet_metal` and `08_resolution_C6_vial`, which are identical except for the product.

**Runs on:** Kaggle with the GPU and internet on, to install anomalib and fetch the pretrained network. On the laptop it only checks the code, with an untrained network, a tiny image size and a few sample photos, so its numbers mean nothing.

**Settings** come from `configs/C6_*.yaml`, and **all measuring** from `src/evaluate.py`, so every configuration is measured the same way. On Kaggle, both come from the private dataset `revathyh/visual-inspection-src`.

**Half B is never read.** Half A's part numbers are written here because Kaggle cannot read files from the laptop, and the code stops if the photo count is wrong.

**Figures with dataset images:** files named `gallery_*` show real parts, so they stay on the laptop and out of git. Each figure is saved and closed, not displayed.

In [ ]:
!pip install anomalib==2.7.0

## 1. Settings and code

`IMAGE_SIZE` is the only setting that differs from Phase 7.

In [ ]:
CONFIG_ID = "C6"

import csv
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy
import torch
import yaml
from anomalib.data import MVTecAD2
from anomalib.models import Padim
from anomalib.models.image.padim.torch_model import PadimModel
from anomalib.models.image.patchcore.torch_model import PatchcoreModel
from torch.utils.data import DataLoader, Subset

USING_GPU = torch.cuda.is_available()

if USING_GPU:
    DEVICE = "cuda"
    INPUT_FOLDER = Path("/kaggle/input")
    PRETRAINED = True
    EXPECTED_HALF_A_PHOTOS = {"sheet_metal": 54, "vial": 70}
else:
    # The laptop only proves the code runs: tiny photos and an untrained network
    DEVICE = "cpu"
    INPUT_FOLDER = Path("data/sample")
    PRETRAINED = False
    EXPECTED_HALF_A_PHOTOS = {"sheet_metal": 2, "vial": 4}

# Finds the project code and settings: the private dataset on Kaggle, or the project folder on the laptop
code_folder = None
for evaluate_file in Path("/kaggle/input").rglob("evaluate.py") if USING_GPU else [Path("src/evaluate.py")]:
    code_folder = evaluate_file.parent.parent
if code_folder is None:
    raise FileNotFoundError("Could not find src/evaluate.py")
sys.path.insert(0, str(code_folder))
from src import evaluate

config_files = list((code_folder / "configs").glob(CONFIG_ID + "_*.yaml"))
if len(config_files) != 1:
    raise FileNotFoundError("Expected one settings file for " + CONFIG_ID)
with open(config_files[0], encoding="utf-8") as file:
    config = yaml.safe_load(file)
method_settings = config["method"]
print("Settings file:", config_files[0].name)
print("Method:", method_settings["name"], "| network:", method_settings["backbone"], "| seeds:", config["seeds"])

# The only change from Phase 7: the resolution test size, 512
IMAGE_SIZE = config["image"]["resolution_test_size"] if USING_GPU else 64
# Added to every file name, so 512 results never mix with the 256 ones
RUN_TAG = CONFIG_ID + "_512"
BATCH_SIZE = method_settings["train_batch_size"]

# Half A, from half_a_files.csv. Every lighting version of each part is used
HALF_A_PARTS = {
    "sheet_metal": {"good": ["001", "002"], "bad": ["000", "004", "005", "007", "009", "012", "013"]},
    "vial": {"good": ["000", "002", "003"], "bad": ["001", "008", "010", "011", "012", "013", "014"]},
}
# Heat maps are drawn for fixed photos chosen in advance, so they cannot flatter a result
HEAT_MAP_PARTS = {
    "sheet_metal": [("bad", "000"), ("bad", "004"), ("bad", "009"), ("good", "001")],
    "vial": [("bad", "001"), ("bad", "010"), ("bad", "011"), ("good", "000")],
}
FIGURE_DPI = 150
RESULTS_FOLDER = Path("results")
RESULTS_FOLDER.mkdir(exist_ok=True)

## 2. Helper functions

- `set_seed` fixes randomness for Python, NumPy and PyTorch. PatchCore's coreset starts from a random patch, using PyTorch's random numbers, and its quick random squeeze uses scikit-learn, which falls back on NumPy's random numbers when no seed is given. PaDiM's random choice of features uses PyTorch's.
- `build_model` builds the method named in the settings file, with the settings file's network and layers.
- `fit` learns from good training photos. PaDiM collects features on the GPU, then runs its statistics on the processor, because the GPU's many-at-once inverse crashes in this setup. PatchCore collects features, then keeps the most varied 10 percent, called the coreset.
- `score_photos` runs the model on photos. Scores are reshaped into a simple list with `reshape(-1)`, because a column of scores compared with a list of labels gives wrong results without an error.
- `photo_fingerprint` sums PatchCore's memory bank, a quick way to check that different seeds really chose different patches.

In [ ]:
def set_seed(seed):
    """Fix the random numbers used by Python, NumPy and PyTorch."""
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)


def build_model():
    """Build the model named in the settings file."""
    if method_settings["name"] == "padim":
        model = PadimModel(backbone=method_settings["backbone"], layers=method_settings["layers"],
                           pre_trained=PRETRAINED, n_features=method_settings["n_features"])
    elif method_settings["name"] == "patchcore":
        model = PatchcoreModel(backbone=method_settings["backbone"], layers=method_settings["layers"],
                               pre_trained=PRETRAINED, num_neighbors=method_settings["num_neighbors"])
    else:
        raise ValueError("This notebook runs only PaDiM and PatchCore")
    return model.to(DEVICE)


def make_loader(dataset, positions=None):
    """Build batches in a fixed order, optionally from chosen photos only."""
    source = dataset
    if positions is not None:
        source = Subset(dataset, positions)
    return DataLoader(source, batch_size=BATCH_SIZE, collate_fn=dataset.collate_fn, shuffle=False)


def fit(model, train_data, recipe):
    """Learn from good training photos."""
    model.train()
    # The pretrained network must stay unchanged while features are collected
    model.feature_extractor.eval()
    with torch.no_grad():
        for batch in make_loader(train_data):
            model(recipe(batch.image).to(DEVICE))

    if method_settings["name"] == "padim":
        embeddings = torch.vstack(model.memory_bank).cpu()
        model.memory_bank = []
        # anomalib's own statistics, run on the processor
        model.gaussian.fit(embeddings)
        model.gaussian.to(DEVICE)
    else:
        # Keeps the most varied share of patches, anomalib's default of 10 percent
        model.subsample_embedding(method_settings["coreset_sampling_ratio"])
    model.eval()


def score_photos(model, loader, recipe):
    """Run the model on photos. Return maps, scores, masks, paths and model seconds."""
    maps = []
    scores = []
    masks = []
    paths = []
    model_seconds = 0.0
    with torch.no_grad():
        for batch in loader:
            images, batch_masks = recipe(batch.image, batch.gt_mask)
            images = images.to(DEVICE)
            started = evaluate.start_timer()
            output = model(images)
            model_seconds = model_seconds + evaluate.stop_timer(started)
            # The map has an extra dimension for its one channel
            maps.append(output.anomaly_map.squeeze(1).cpu())
            # Turns the scores into a simple list, to line up with the labels
            scores.append(output.pred_score.reshape(-1).cpu())
            masks.append(batch_masks)
            paths.extend(batch.image_path)
    return torch.cat(maps), torch.cat(scores), torch.cat(masks), paths, model_seconds


def photo_fingerprint(model):
    """Sum PatchCore's memory bank, to check that seeds chose different patches."""
    if method_settings["name"] == "patchcore":
        return round(float(model.memory_bank.sum()), 4)
    return "not applicable"

## 3. Run every product and seed

For each product and seed: learn from training photos, set the benchmark threshold from good validation photos, then score half A. One extra pass comes before timing, because the first pass includes set-up work.

Each run's results are saved straight away, so a later problem cannot lose them.

**New in Phase 7:** for the first seed, a failure gallery is also drawn, the same as in Phase 6: the 2 defective photos with the highest scores, the 4 defective photos with the lowest, and the 4 good photos with the highest. They are picked by score, not by where a line sits. Drawing them here means no separate rerun is needed to look at failures.

In [ ]:
recipe = Padim.configure_pre_processor(image_size=(IMAGE_SIZE, IMAGE_SIZE)).transform
summary_rows = []
failure_rows = []

# One product per notebook, so both can run at the same time
for product_name in ["vial"]:
    product_folder = None
    for train_folder in INPUT_FOLDER.rglob("train"):
        if train_folder.parent.name == product_name:
            product_folder = train_folder.parent
    if product_folder is None:
        raise FileNotFoundError("Could not find " + product_name)

    # prepare_data is not called, as it could try to download
    datamodule = MVTecAD2(root=product_folder.parent, category=product_name, num_workers=0)
    datamodule.setup()

    half_a_positions = []
    test_paths = datamodule.test_public_data.samples["image_path"].tolist()
    for position in range(len(test_paths)):
        path = Path(test_paths[position])
        if path.stem.split("_")[0] in HALF_A_PARTS[product_name][path.parent.name]:
            half_a_positions.append(position)
    if len(half_a_positions) != EXPECTED_HALF_A_PHOTOS[product_name]:
        raise ValueError("Wrong number of half A photos for " + product_name + ": " + str(len(half_a_positions)))

    for seed in config["seeds"]:
        run_name = RUN_TAG + "_" + product_name + "_seed" + str(seed)
        print("===", run_name)
        set_seed(seed)
        model = build_model()

        evaluate.reset_peak_memory()
        started = evaluate.start_timer()
        fit(model, datamodule.train_data, recipe)
        fit_seconds = evaluate.stop_timer(started)
        peak_fit = evaluate.peak_memory_gb()

        with torch.no_grad():
            model(torch.zeros(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE))

        validation_maps, validation_scores, ignored_masks, ignored_paths, ignored_seconds = score_photos(
            model, make_loader(datamodule.val_data), recipe)

        evaluate.reset_peak_memory()
        maps, scores, masks, paths, model_seconds = score_photos(
            model, make_loader(datamodule.test_public_data, half_a_positions), recipe)
        peak_scoring = evaluate.peak_memory_gb()

        labels = []
        for path in paths:
            labels.append(Path(path).parent.name == "bad")
        labels = torch.tensor(labels)

        results = evaluate.evaluate(validation_maps, maps, scores, masks, labels, model_seconds, peak_scoring)
        threshold = results["threshold"]
        results["config"] = CONFIG_ID
        results["product"] = product_name
        results["seed"] = seed
        results["half_a_photos"] = len(paths)
        results["defective_photos"] = int(labels.sum())
        results["good_validation_photos_flagged"] = int((validation_scores > threshold).sum())
        results["good_validation_photos"] = len(validation_scores)
        results["good_half_a_photos_flagged"] = int(((scores > threshold) & ~labels).sum())
        results["defective_half_a_photos_flagged"] = int(((scores > threshold) & labels).sum())
        results["fit_seconds"] = round(fit_seconds, 2)
        results["peak_gpu_gb_fit"] = peak_fit
        results["memory_bank_fingerprint"] = photo_fingerprint(model)
        results["pretrained"] = PRETRAINED
        results["image_size"] = IMAGE_SIZE
        # Secondary metrics for the appendix, new in Phase 8
        results["image_auprc"] = evaluate.image_auprc(scores, labels)
        results["pixel_average_precision"] = evaluate.pixel_average_precision(maps, masks)

        # Numbers only, with no image content, so these files can be committed
        with open(RESULTS_FOLDER / (run_name + "_metrics.json"), "w") as file:
            json.dump(results, file, indent=1)
        with open(RESULTS_FOLDER / (run_name + "_photo_scores.csv"), "w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["file_name", "label", "score", "above_threshold"])
            for position in range(len(paths)):
                writer.writerow([Path(paths[position]).name, Path(paths[position]).parent.name,
                                 round(float(scores[position]), 4), bool(scores[position] > threshold)])
        summary_rows.append(results)
        for key in results:
            print(" ", key, results[key])

        # Heat maps for the first seed only, for fixed photos
        if seed == config["seeds"][0]:
            chosen = []
            for position in range(len(paths)):
                path = Path(paths[position])
                for label, part in HEAT_MAP_PARTS[product_name]:
                    if path.parent.name == label and path.name == part + "_regular.png":
                        chosen.append(position)
            if len(chosen) > 0:
                normalize_step = recipe.transforms[1]
                means = torch.tensor(normalize_step.mean).view(3, 1, 1)
                stds = torch.tensor(normalize_step.std).view(3, 1, 1)
                top = float(maps.max())
                bottom = float(maps.min())
                figure, axes = plt.subplots(len(chosen), 2, figsize=(9, 4.2 * len(chosen)), squeeze=False)
                for row_number in range(len(chosen)):
                    position = chosen[row_number]
                    photo = recipe(datamodule.test_public_data[half_a_positions[position]].image)
                    photo = (photo * stds + means).clamp(0, 1)
                    mask = masks[position].float()
                    axes[row_number][0].imshow(photo[0], cmap="gray", vmin=0, vmax=1)
                    axes[row_number][1].imshow(maps[position], cmap="Blues", vmin=bottom, vmax=top)
                    if bool(mask.any()):
                        axes[row_number][0].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
                        axes[row_number][1].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
                    if float(maps[position].max()) > threshold:
                        axes[row_number][1].contour(maps[position], levels=[threshold], colors="black", linewidths=0.8)
                    axes[row_number][0].set_title(Path(paths[position]).parent.name + " " + Path(paths[position]).name, fontsize=10)
                    axes[row_number][1].set_title("Score " + str(round(float(scores[position]), 2)) + ", threshold " + str(round(threshold, 2)), fontsize=10)
                    axes[row_number][0].axis("off")
                    axes[row_number][1].axis("off")
                figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0), half A, " + RUN_TAG
                                + ". Red: mask. Black: above threshold", y=1.0)
                figure.tight_layout()
                figure.savefig("gallery_heatmaps_" + run_name + ".png", dpi=FIGURE_DPI, bbox_inches="tight")
                # Closing without showing keeps dataset images out of the notebook's log
                plt.close(figure)

            # Failure gallery, picked by score as in Phase 6
            defective = []
            good = []
            for position in range(len(paths)):
                if labels[position]:
                    defective.append(position)
                else:
                    good.append(position)

            def score_of(position):
                return float(scores[position])

            defective.sort(key=score_of)
            good.sort(key=score_of)
            picked = []
            for position in defective[-2:]:
                picked.append(("hit", position))
            for position in defective[:4]:
                picked.append(("hardest defective", position))
            for position in good[-4:]:
                picked.append(("hardest good", position))
            normalize_step = recipe.transforms[1]
            means = torch.tensor(normalize_step.mean).view(3, 1, 1)
            stds = torch.tensor(normalize_step.std).view(3, 1, 1)
            top = float(maps.max())
            bottom = float(maps.min())
            figure, axes = plt.subplots(len(picked), 2, figsize=(8, 3.6 * len(picked)), squeeze=False)
            for row_number in range(len(picked)):
                kind, position = picked[row_number]
                path = Path(paths[position])
                photo = recipe(datamodule.test_public_data[half_a_positions[position]].image)
                photo = (photo * stds + means).clamp(0, 1)
                mask = masks[position].float()
                axes[row_number][0].imshow(photo[0], cmap="gray", vmin=0, vmax=1)
                axes[row_number][1].imshow(maps[position], cmap="Blues", vmin=bottom, vmax=top)
                if bool(mask.any()):
                    axes[row_number][0].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
                    axes[row_number][1].contour(mask, levels=[0.5], colors="red", linewidths=0.8)
                axes[row_number][0].set_title(kind + ": " + path.parent.name + " " + path.name, fontsize=9)
                axes[row_number][1].set_title("score " + str(round(float(scores[position]), 2)), fontsize=9)
                axes[row_number][0].axis("off")
                axes[row_number][1].axis("off")
                failure_rows.append([RUN_TAG, product_name, kind, path.parent.name, path.name,
                                     round(float(scores[position]), 4)])
            figure.suptitle("MVTec AD 2, " + product_name + " (CC BY-NC-SA 4.0), half A, " + RUN_TAG
                            + ", seed " + str(seed) + ". Red: mask. Darker blue: more suspicious", y=1.0)
            figure.tight_layout()
            figure.savefig("gallery_failures_" + RUN_TAG + "_" + product_name + ".png", dpi=120, bbox_inches="tight")
            plt.close(figure)
            # Numbers only, saved after each product so a disconnect cannot lose them
            with open(RESULTS_FOLDER / (RUN_TAG + "_" + product_name + "_failure_picks.csv"), "w", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(["config", "product", "kind", "label", "file_name", "score"])
                for row in failure_rows:
                    writer.writerow(row)

        del model
        evaluate.reset_peak_memory()

with open(RESULTS_FOLDER / (RUN_TAG + "_" + product_name + "_summary.csv"), "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    for row in summary_rows:
        writer.writerow(row)
print("Finished", CONFIG_ID)